# 03 — Compare All Controllers

Evaluates every controller under **identical** conditions:

| Controller | Type |
|---|---|
| No cooling | Baseline |
| Constant u=0.5 | Baseline |
| Constant u=1.0 | Baseline |
| Bang-bang | Classical |
| Proportional | Classical |
| PI tuned | Classical |
| PPO | RL |
| SAC | RL |

All controllers run on the same 4 heat profiles with the same random seed.

**Run training notebooks first** so models exist on Drive.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust path if needed
%cd /content/drive/MyDrive/RL-Battery-Thermal-Management

import os, sys
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

!pip install -q -r requirements.txt
print('Setup complete.')

## Model and Results Paths

In [ ]:
BASE_DRIVE_DIR = '/content/drive/MyDrive/battery_rl'

PPO_MODEL   = f'{BASE_DRIVE_DIR}/models/ppo_3d_pack/ppo_pack_final.zip'
PPO_VECNORM = f'{BASE_DRIVE_DIR}/models/ppo_3d_pack/vec_normalize.pkl'
SAC_MODEL   = f'{BASE_DRIVE_DIR}/models/sac_3d_pack/sac_pack_final.zip'
RESULTS_DIR = f'{BASE_DRIVE_DIR}/results/comparison'

os.makedirs(RESULTS_DIR, exist_ok=True)

# Report what exists
for label, path in [('PPO model', PPO_MODEL), ('PPO vecnorm', PPO_VECNORM),
                     ('SAC model', SAC_MODEL)]:
    status = '✓ found' if os.path.exists(path) else '✗ missing'
    print(f'{status}  {label}: {path}')

## Run Comparison

`--episodes 20` runs 5 rounds of 4 heat profiles each (same seed per round).
If a model is missing, that controller is silently skipped and only baselines run.

In [ ]:
!python evaluation/compare_controllers.py \
  --ppo-model   "$PPO_MODEL" \
  --ppo-vecnorm "$PPO_VECNORM" \
  --sac-model   "$SAC_MODEL" \
  --results-dir "$RESULTS_DIR" \
  --episodes 20 \
  --seed 7

## Results

In [ ]:
import pandas as pd

print('Output files:')
for f in sorted(os.listdir(RESULTS_DIR)):
    size_kb = os.path.getsize(os.path.join(RESULTS_DIR, f)) // 1024
    print(f'  {f}  ({size_kb} KB)')

In [ ]:
csv_path = os.path.join(RESULTS_DIR, 'comparison_metrics.csv')
df = pd.read_csv(csv_path)

summary = (
    df.groupby('controller')[['total_reward', 'T_max_peak_C',
                               'T_gradient_max_C', 'time_above_safe_s',
                               'total_cooling_effort']]
      .mean()
      .sort_values('total_reward', ascending=False)
      .round(2)
)
print('Mean metrics across all profiles and rounds:\n')
print(summary.to_string())

## Display Plots

In [ ]:
from IPython.display import Image, display
import glob

plot_files = sorted(glob.glob(os.path.join(RESULTS_DIR, '*.png')))
print(f'Displaying {len(plot_files)} plots:')
for p in plot_files:
    print(' ', os.path.basename(p))
    display(Image(p))

## Baselines Only (no RL models required)

Useful to verify the classical controller benchmark before training.

In [ ]:
# Uncomment to run classical controllers only (no RL models needed):
# BASELINE_RESULTS = f'{BASE_DRIVE_DIR}/results/baselines_only'
# os.makedirs(BASELINE_RESULTS, exist_ok=True)
#
# !python evaluation/compare_controllers.py \
#   --results-dir "$BASELINE_RESULTS" \
#   --episodes 4 \
#   --seed 7